In [ ]:
!uv pip install -r requirements.txt

In [ ]:
# Import the model loader
import sys
sys.path.append('/workspaces/embedding_benchmark')
import json
import csv

from utils.config import load_embedding_model

In [ ]:
# Define models to evaluate
models = [
    "nvidia/llama-nemotron-embed-1b-v2",
    "Qwen/Qwen3-Embedding-4B",
    "Linq-AI-Research/Linq-Embed-Mistral",
    "intfloat/multilingual-e5-large-instruct",
    "intfloat/e5-mistral-7b-instruct",
    "Qwen/Qwen3-Embedding-0.6B",
    "sentence-transformers/all-MiniLM-L6-v2",
    "intfloat/e5-small-v2",
    "intfloat/e5-base-instruct",
    "intfloat/e5-large-instruct",
]

# Load corpus chunks (raw text)
with open("/workspaces/embedding_benchmark/data/processed/chunks.jsonl", "r") as f:
    chunks = [json.loads(line) for line in f.readlines()]
    chunk_ids = [chunk['id'] for chunk in chunks]
    texts = [chunk['text'] for chunk in chunks]

# Load test queries (raw text)
with open("/workspaces/embedding_benchmark/data/processed/test_queries.csv") as f:
    csv_reader = list(csv.reader(f))[1:]  # Skip header
    queries = [row[1] for row in csv_reader]

print(f"Loaded {len(chunks)} chunks and {len(queries)} queries")

In [ ]:
def format_texts_for_model(model_id: str, texts: list, text_type: str = "passage"):
    """
    Format texts according to model-specific requirements.
    
    Args:
        model_id: Hugging Face model ID
        texts: List of text strings to format
        text_type: Either "query" or "passage" to determine formatting
    
    Returns:
        List of formatted text strings
    """
    # Instruction-based models (e5-instruct variants, Mistral-based, Qwen)
    instruction_models = [
        "multilingual-e5-large-instruct",
        "e5-mistral-7b-instruct", 
        "e5-large-instruct",
        "linq-embed-mistral",
        "qwen3-embedding"
    ]
    
    # Prefix-based models (older e5 models)
    prefix_models = [
        "e5-small-v2",
        "e5-base-instruct"
    ]
    
    model_lower = model_id.lower()
    
    # Check if instruction-based model
    if any(inst_model in model_lower for inst_model in instruction_models):
        if text_type == "query":
            # Add instruction prefix for queries
            instruction = "Instruct: Given a web search query, retrieve relevant passages that answer the query\nQuery: "
            return [instruction + text for text in texts]
        else:
            # No prefix for corpus/passages
            return texts
    
    # Check if prefix-based model
    elif any(prefix_model in model_lower for prefix_model in prefix_models):
        if text_type == "query":
            return ["query: " + text for text in texts]
        else:
            return ["passage: " + text for text in texts]
    
    # Raw text models (all-MiniLM, Nemotron, etc.)
    else:
        return texts

In [ ]:
import numpy as np

# Dictionary to store embeddings for each model
model_embeddings = {}

for model_id in models:
    try:
        print(f"\n{'='*60}")
        print(f"Processing: {model_id}")
        print(f"{'='*60}")
        
        # Load the model
        model = load_embedding_model(model_id)
        print(f"✓ Model loaded successfully")
        
        # Format texts according to model requirements
        formatted_corpus = format_texts_for_model(model_id, texts, text_type="passage")
        formatted_queries = format_texts_for_model(model_id, queries, text_type="query")
        
        print(f"✓ Formatted {len(formatted_corpus)} corpus texts")
        print(f"✓ Formatted {len(formatted_queries)} queries")
        
        # Encode corpus
        print("Encoding corpus...")
        corpus_embeddings = model.encode(formatted_corpus, batch_size=32, show_progress_bar=True)
        
        # Encode queries
        print("Encoding queries...")
        query_embeddings = model.encode(formatted_queries, batch_size=32, show_progress_bar=True)
        
        # Store embeddings with metadata
        model_embeddings[model_id] = {
            'corpus_embeddings': corpus_embeddings,
            'query_embeddings': query_embeddings,
            'chunk_ids': chunk_ids,
            'embedding_dim': corpus_embeddings.shape[1]
        }
        
        print(f"✓ Corpus embeddings shape: {corpus_embeddings.shape}")
        print(f"✓ Query embeddings shape: {query_embeddings.shape}")
        
    except Exception as e:
        print(f"✗ Error processing model {model_id}: {str(e)}")
        continue

print(f"\n{'='*60}")
print(f"Successfully processed {len(model_embeddings)} models")
print(f"{'='*60}")